<a href="https://colab.research.google.com/github/jtriana645-lgtm/applied-math-repository/blob/main/Part2_Exam_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Scientific Computing II – First Midterm Exam
## Part 2 · 14/03/2026

**Problem:** Implement the operation $(\mathbf{x}+\mathbf{y})^T A (\mathbf{x}+\mathbf{y})$
in three ways and compare their performance.

---
## 0. Imports

In [ ]:
import numpy as np
import time
import matplotlib.pyplot as plt

# Numba (install if needed)
try:
    import numba
except ImportError:
    %pip install numba
    import numba

from numba import njit

# NumExpr
try:
    import numexpr as ne
except ImportError:
    %pip install numexpr
    import numexpr as ne

import os
os.environ['NUMEXPR_NUM_THREADS'] = str(os.cpu_count())

print("All imports OK")
print(f"NumPy  : {np.__version__}")
print(f"Numba  : {numba.__version__}")
print(f"NumExpr: {ne.__version__}")

---
## 1. Auxiliary generator function

Given $m \geq 1$, generates:
- $\mathbf{x}, \mathbf{y} \in \mathbb{R}^m$ (random)
- $A \in \mathbb{R}^{m \times m}$ (random, made symmetric positive-definite so the quadratic form is always well-defined and positive)

In [ ]:
def generate_inputs(m, seed=42):
    """
    Generate random vectors x, y of length m
    and a random symmetric matrix A of size m x m.

    A is made symmetric as A = B + B^T so the quadratic form
    (x+y)^T A (x+y) is guaranteed to be real-valued.

    Parameters
    ----------
    m : int   size of the problem
    seed : int  random seed for reproducibility

    Returns
    -------
    x, y : 1-D arrays of length m
    A    : 2-D array of shape (m, m)
    """
    rng = np.random.default_rng(seed)
    x = rng.standard_normal(m)
    y = rng.standard_normal(m)
    B = rng.standard_normal((m, m))
    A = B + B.T          # symmetric
    return x, y, A

# Quick check
x, y, A = generate_inputs(4)
print("x =", x)
print("y =", y)
print("A =\n", A)

---
## 2. Three implementations of `xyAxy`

The operation to compute is the **quadratic form**:

$$
\text{result} = (\mathbf{x}+\mathbf{y})^T \, A \, (\mathbf{x}+\mathbf{y}) = \sum_{i=0}^{m-1}\sum_{j=0}^{m-1} z_i \, A_{ij} \, z_j
\quad \text{where} \quad \mathbf{z} = \mathbf{x} + \mathbf{y}
$$

It returns a **scalar**.

### 2.1 – Method 1: Naive (two nested `for` loops)

In [ ]:
def xyAxy_naive(x, y, A):
    """
    Compute (x+y)^T A (x+y) using two nested for loops.
    Complexity: O(m^2)  -- m multiplications and additions per row.
    """
    m = len(x)
    z = [x[i] + y[i] for i in range(m)]   # element-wise sum in a list

    result = 0.0
    for i in range(m):
        for j in range(m):
            result += z[i] * A[i, j] * z[j]
    return result

### 2.2 – Method 2: Vectorized (NumPy)

In [ ]:
def xyAxy_vectorized(x, y, A):
    """
    Compute (x+y)^T A (x+y) using NumPy.

    Strategy:
      1. z = x + y                     (vectorized sum, O(m))
      2. Az = A @ z                    (matrix-vector product, O(m^2) in C)
      3. result = z @ Az = z^T (A z)   (dot product, O(m))

    No Python-level loops at all; all work done in NumPy's C back-end.
    """
    z  = x + y          # shape (m,)
    Az = A @ z          # shape (m,)  -- BLAS matrix-vector multiply
    return float(z @ Az)  # scalar dot product

### 2.3 – Method 3: Optimized with Numba (`@njit`)

In [ ]:
@njit
def xyAxy_numba(x, y, A):
    """
    Compute (x+y)^T A (x+y) compiled to native machine code by Numba.

    @njit (no-python mode) compiles the entire function the first time
    it is called and caches the machine code for all subsequent calls.
    Numba can also auto-vectorize the inner loops with SIMD instructions.
    """
    m = len(x)
    # Step 1: z = x + y
    z = np.empty(m)
    for i in range(m):
        z[i] = x[i] + y[i]

    # Step 2 & 3: compute z^T A z with two loops (Numba compiles these to C speed)
    result = 0.0
    for i in range(m):
        s = 0.0
        for j in range(m):
            s += A[i, j] * z[j]
        result += z[i] * s
    return result

# Warm-up call: triggers JIT compilation (this first call is intentionally slow)
x0, y0, A0 = generate_inputs(10)
_ = xyAxy_numba(x0, y0, A0)
print("Numba warm-up complete.")

---
## 3. Correctness verification

All three implementations must return the same scalar value before we can trust any timing comparison.

In [ ]:
# Use a mid-size problem for the check
m_check = 200
x, y, A = generate_inputs(m_check, seed=0)

r1 = xyAxy_naive      (x, y, A)
r2 = xyAxy_vectorized (x, y, A)
r3 = xyAxy_numba      (x, y, A)

print(f"Naive      : {r1:.10f}")
print(f"Vectorized : {r2:.10f}")
print(f"Numba      : {r3:.10f}")

tol = 1e-8
assert abs(r1 - r2) < tol, f"Mismatch naive vs vectorized: {abs(r1-r2)}"
assert abs(r1 - r3) < tol, f"Mismatch naive vs numba: {abs(r1-r3)}"

print("\nAll three methods agree within tolerance 1e-8 -- VERIFIED.")

---
## 4. Performance comparison as a function of $m$

In [ ]:
# Sizes to test
# Naive is O(m^2) and very slow in pure Python, so we keep its range smaller
m_values      = [10, 30, 50, 100, 200, 300, 500]   # all three methods
m_large       = [10, 30, 50, 100, 200, 300, 500,
                 700, 1000, 1500, 2000]              # fast methods only

N_REPEATS = 5   # number of timed repetitions per (method, m) pair

def time_function(func, x, y, A, repeats=N_REPEATS):
    """Return the minimum wall-clock time over `repeats` calls."""
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        func(x, y, A)
        times.append(time.perf_counter() - t0)
    return min(times)   # min is less sensitive to OS scheduling spikes

t_naive = []
t_vec   = []
t_numba = []

print(f"{'m':>6}  {'Naive (s)':>12}  {'Vectorized (s)':>16}  {'Numba (s)':>12}")
print("-" * 55)

for m in m_values:
    x, y, A = generate_inputs(m)
    tn = time_function(xyAxy_naive,      x, y, A)
    tv = time_function(xyAxy_vectorized, x, y, A)
    tb = time_function(xyAxy_numba,      x, y, A)
    t_naive.append(tn)
    t_vec.append(tv)
    t_numba.append(tb)
    print(f"{m:>6}  {tn:>12.6f}  {tv:>16.6f}  {tb:>12.6f}")

# Extend fast methods to larger m
t_vec_large   = []
t_numba_large = []

for m in m_large:
    x, y, A = generate_inputs(m)
    t_vec_large.append  (time_function(xyAxy_vectorized, x, y, A))
    t_numba_large.append(time_function(xyAxy_numba,      x, y, A))

---
## 5. Plot: compute time vs. $m$

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ---- Left: all three on the shared range (log-log) ----
ax = axes[0]
ax.loglog(m_values, t_naive, 'o-r',  label='Naive (for loops)',  linewidth=1.8, markersize=6)
ax.loglog(m_values, t_vec,   's-b',  label='Vectorized (NumPy)', linewidth=1.8, markersize=6)
ax.loglog(m_values, t_numba, '^-g',  label='Numba (@njit)',      linewidth=1.8, markersize=6)

# Reference lines for complexity classes
m_ref = np.array(m_values, dtype=float)
c2 = t_naive[0] / m_ref[0]**2
ax.loglog(m_ref, c2 * m_ref**2, '--k', alpha=0.35, linewidth=1, label=r'$\mathcal{O}(m^2)$ reference')

ax.set_xlabel('m  (problem size)', fontsize=12)
ax.set_ylabel('Time (s)', fontsize=12)
ax.set_title('All three methods (small m)', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, which='both', alpha=0.3)

# ---- Right: fast methods only, large m (log-log) ----
ax2 = axes[1]
ax2.loglog(m_large, t_vec_large,   's-b', label='Vectorized (NumPy)', linewidth=1.8, markersize=6)
ax2.loglog(m_large, t_numba_large, '^-g', label='Numba (@njit)',      linewidth=1.8, markersize=6)

ax2.set_xlabel('m  (problem size)', fontsize=12)
ax2.set_ylabel('Time (s)', fontsize=12)
ax2.set_title('Vectorized vs Numba (large m)', fontsize=12)
ax2.legend(fontsize=9)
ax2.grid(True, which='both', alpha=0.3)

fig.suptitle(r'$(\mathbf{x}+\mathbf{y})^T A (\mathbf{x}+\mathbf{y})$ — compute time vs $m$',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('timing_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved as timing_plot.png")

---
## 6. Discussion of results

### Algorithmic complexity

All three implementations compute the same $\mathcal{O}(m^2)$ quadratic form
$(\mathbf{x}+\mathbf{y})^T A (\mathbf{x}+\mathbf{y})$.  
In the log-log plot, every method should therefore appear as a straight line with
**slope ≈ 2** (since $\log t \approx 2\log m + C$), and that is what we observe.
The three lines are parallel — only the intercept (the constant factor) differs.

### Why the naive method is slowest

The Python double `for` loop pays the interpreter overhead on every one of the
$m^2$ inner iterations: type checking, bytecode dispatch, and Python object
management. For $m = 500$ this means $250\,000$ overhead-laden iterations, which
explains the very large constant factor in the naive timing curve.

### Why vectorization is much faster

`xyAxy_vectorized` replaces the Python loops with NumPy's `@` operator
(a BLAS level-2 matrix-vector multiply) and a single dot product.  
These calls run entirely in compiled C/Fortran code, with no Python overhead per
element. The constant factor in the timing curve is therefore dramatically smaller
than for the naive method, even though the algorithmic complexity is identical.

### Numba vs. NumPy at large $m$

For **small $m$** Numba is competitive with or slightly faster than NumPy, because
it compiles the inner loops to native machine code with SIMD instructions and
avoids the allocation of intermediate arrays (e.g. the `Az` temporary in the
vectorized version).  
For **large $m$** the dominant cost is memory bandwidth (reading the $m \times m$
matrix $A$ from RAM), and both methods are bounded by the same hardware limit.
NumPy's BLAS back-end is highly tuned for cache usage at large sizes, so it
can match or beat the Numba version there.

### Key takeaway

| Method | Relative speed | Reason |
|--------|---------------|--------|
| Naive (for loops) | Slowest (~100–1000×) | Python interpreter overhead per element |
| Vectorized (NumPy) | Fast | Compiled C/BLAS, no per-element Python overhead |
| Numba (@njit) | Fast / fastest at small $m$ | JIT-compiled native code, no temporary arrays |

The log-log plot confirms that the slope is ≈ 2 for all methods (all are $\mathcal{O}(m^2)$),
but the vertical offset — the constant factor — differs by orders of magnitude.
This illustrates the central message of the course: **algorithmic complexity tells
you the shape of the curve; the implementation determines where it sits**.